# Проект 1. Что случилось с выручкой ПРАЙМ в августе 2026?

## Бизнес-вопрос
Почему в августе 2026 года остановился рост выручки от подписки и возник
шестикратный всплеск оттока — и является ли это проблемой продукта
или следствием отдельного операционного события?

## Гипотезы (в порядке проверки)
H1 (первичная). Сезонная финансовая нагрузка: отпуска + подготовка к школе →
рост отказов `too_expensive` и падение платёжеспособности.
H2. Отложенная реакция на повышение тарифов 01.03.2026.
H3 (рабочая, сформирована по данным). Пакетное закрытие подписок одного
канала привлечения 22–23.08.2026, а не органический отток.

## Зерно расчёта и почему именно оно
- **Выручка:** одна строка = один УСПЕШНЫЙ платёж (`payments.payment_id`,
  `status = 'success'`). Деньги списывает владелец (`subscriptions.owner_client_id`),
  поэтому `subscription_members` в расчёте выручки НЕ участвует:
  через участников платёж размножится на каждого члена семьи,
  а подписки без участников исчезнут при INNER JOIN.
- **Декомпозиция:** одна строка = месяц × тариф × флаг аномальной когорты.
  Тариф нужен, потому что с марта цены разные; сегмент нужен,
  потому что он и есть подозреваемый фактор.
- **Период:** 01.07–28.07.2026 против 01.08–28.08.2026 (по 28 дней).
  Август в выгрузке неполный — сравнивать его целиком с июлем нельзя.
- **Время:** все даты в UTC (`set time zone 'UTC'` в `query`),
  локальные часовые пояса не смешиваем: вопрос про кассовую выручку компании,
  не про локальный день клиента.
- **Деньги:** агрегаты `SUM(numeric)` считаются в PostgreSQL,
  pandas используется только для отрисовки.


In [ ]:
import sys
from collections import Counter, defaultdict  # понадобятся в задачах
from datetime import date
from decimal import Decimal

from sqlalchemy import text

# get_engine() живёт в вашем репозитории, в src/prime/config.py.
# Она читает строку подключения из .env — поэтому пароля здесь нет.
try:
    from prime.config import get_engine
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Пакет prime не найден: ноутбук запущен не в окружении prime-monitor.\n"
        f"Сейчас Python отсюда: {sys.executable}\n"
        "JupyterLab: закройте его, перейдите в корень prime-monitor и выполните uv run jupyter lab.\n"
        "VS Code: Select Kernel справа сверху -> .venv из папки prime-monitor."
    ) from None

engine = get_engine()


def query(sql: str, **params) -> list[dict]:
    """Выполнить SQL-запрос и вернуть строки списком словарей {колонка: значение}."""
    try:
        with engine.connect() as conn:
            # Моменты времени в базе хранятся с часовым поясом. День события считаем
            # по UTC — так он у всех одинаковый, какие бы настройки ни стояли у вас.
            conn.execute(text("set time zone 'UTC'"))
            result = conn.execute(text(sql), params)
            return [dict(row) for row in result.mappings()]
    finally:
        # Соединение не держим: сервер у всей группы общий, а после сна ноутбука
        # старое соединение всё равно мёртвое.
        engine.dispose()


print("подключились к базе как:", query("select current_user as login")[0]["login"])

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine, text

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titleweight": "bold",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 14,
})

ACCENT = "#C4573A"      # аномалия
NEUTRAL = "#8C9BAB"     # нейтральный
DARK = "#243447"        # итоги
GOOD = "#3E7C6B"        # позитив

def df(sql: str, **params) -> pd.DataFrame:
    """Тот же query, но сразу в DataFrame."""
    return pd.DataFrame(query(sql, **params))

def num(frame: pd.DataFrame, cols) -> pd.DataFrame:
    """Осознанный перевод numeric -> float ТОЛЬКО для графиков/проверок."""
    out = frame.copy()
    for c in cols:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

# Сопоставимые окна: по 28 дней
BASE_START, BASE_END = "2026-07-01", "2026-07-29"
REP_START,  REP_END  = "2026-08-01", "2026-08-29"
BATCH_FROM, BATCH_TO = "2026-08-22", "2026-08-24"
CHANNEL = "partner_telecom"

In [ ]:
# Пересекающиеся интервалы в справочнике цен
dq_tariffs = df("""
    SELECT a.tariff_id,
           a.monthly_fee AS fee_earlier,
           b.monthly_fee AS fee_later,
           GREATEST(a.valid_from, b.valid_from) AS overlap_from,
           LEAST(a.valid_to,   b.valid_to)     AS overlap_to
    FROM prime.tariffs a
    JOIN prime.tariffs b
      ON a.tariff_id = b.tariff_id
     AND a.valid_from < b.valid_from
    WHERE a.valid_from <= b.valid_to
      AND b.valid_from <= a.valid_to
    ORDER BY a.tariff_id, overlap_from;
""")
display(dq_tariffs)

# Двойной счёт через шеринг
dq_join = num(df("""
    WITH correct AS (
        SELECT COUNT(*) AS n_payments, SUM(p.amount) AS revenue
        FROM prime.payments p
        WHERE p.status = 'success'
          AND p.paid_at >= :b_start AND p.paid_at < :r_end
    ), wrong AS (
        SELECT COUNT(*) AS n_rows, SUM(p.amount) AS revenue_if_joined
        FROM prime.payments p
        JOIN prime.subscriptions s ON s.subscription_id = p.subscription_id
        LEFT JOIN prime.subscription_members m
               ON m.subscription_id = s.subscription_id
        WHERE p.status = 'success'
          AND p.paid_at >= :b_start AND p.paid_at < :r_end
    )
    SELECT c.n_payments        AS rows_correct,
           w.n_rows            AS rows_after_members,
           c.revenue           AS revenue_correct,
           w.revenue_if_joined AS revenue_wrong,
           w.revenue_if_joined / NULLIF(c.revenue, 0) AS inflation_factor
    FROM correct c CROSS JOIN wrong w;
""", b_start=BASE_START, r_end=REP_END),
   ["rows_correct", "rows_after_members", "revenue_correct",
    "revenue_wrong", "inflation_factor"])
display(dq_join)

# Целостность связей
dq_integrity = df("""
    SELECT
        (SELECT COUNT(*) FROM prime.payments p
         LEFT JOIN prime.subscriptions s USING (subscription_id)
         WHERE s.subscription_id IS NULL) AS orphan_payments,
        (SELECT COUNT(*) - COUNT(DISTINCT payment_id)
         FROM prime.payments) AS duplicate_payment_ids,
        (SELECT COUNT(*) FROM prime.subscriptions s
         LEFT JOIN prime.clients c ON c.client_id = s.owner_client_id
         WHERE c.client_id IS NULL) AS orphan_subscriptions;
""")
display(dq_integrity)

assert int(dq_integrity.loc[0, "orphan_payments"]) == 0
assert int(dq_integrity.loc[0, "duplicate_payment_ids"]) == 0
print("Вывод: справочник цен требует дедупликации; через members выручка раздувается.")

In [ ]:
monthly = num(df("""
    SELECT date_trunc('month', p.paid_at)::date AS month,
           SUM(p.amount) FILTER (WHERE p.status = 'success')  AS revenue,
           SUM(p.amount) FILTER (WHERE p.status = 'refunded') AS refunded,
           COUNT(*) FILTER (WHERE p.status = 'success')       AS success_payments,
           COUNT(DISTINCT p.subscription_id)
               FILTER (WHERE p.status = 'success')            AS paying_subs,
           COUNT(*) FILTER (WHERE p.status = 'failed')        AS failed_payments
    FROM prime.payments p
    WHERE p.paid_at >= '2025-01-01' AND p.paid_at < '2026-09-01'
    GROUP BY 1
    ORDER BY 1;
"""), ["revenue", "refunded", "success_payments", "paying_subs", "failed_payments"])
monthly["month"] = pd.to_datetime(monthly["month"], utc=True)
display(monthly.tail(8))

# Тренд выручки и платящих подписок
fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                         gridspec_kw={"height_ratios": [2, 1]})

# Верхний график: выручка
axes[0].plot(monthly["month"], monthly["revenue"] / 1e6,
             color=DARK, lw=2, marker="o", ms=4, label="Выручка")
axes[0].set_title("Выручка от подписки: рост до марта, провал гипотезы о «падении продукта» в августе")
axes[0].set_ylabel("Выручка success, млн ₽")
axes[0].axvline(pd.Timestamp("2026-03-01", tz="UTC"), color=NEUTRAL, ls="--", lw=1.5)
axes[0].text(pd.Timestamp("2026-03-05", tz="UTC"), axes[0].get_ylim()[1] * 0.95,
             "Повышение тарифов\n199→249 / 399→449 ₽", fontsize=9, color=NEUTRAL)
axes[0].axvline(pd.Timestamp("2026-08-22", tz="UTC"), color=ACCENT, ls="--", lw=1.5)
axes[0].text(pd.Timestamp("2026-08-22", tz="UTC"), axes[0].get_ylim()[0] * 1.1,
             "22–23 авг:\npакетный отток", fontsize=9, color=ACCENT)
axes[0].legend(loc="upper left")

# Нижний график: платящие подписки
axes[1].plot(monthly["month"], monthly["paying_subs"] / 1000,
             color=NEUTRAL, lw=2, marker="o", ms=3, label="Платящие подписки")
axes[1].set_title("Платящие подписки в месяц (уникальные subscription_id)")
axes[1].set_ylabel("Тыс. подписок")
axes[1].set_xlabel("Месяц, UTC")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# Получаем сырые данные из БД
daily_raw = df("""
    SELECT s.ended_at::date AS day,
           COUNT(*) AS churned,
           COUNT(*) FILTER (WHERE c.acquisition_channel = :ch) AS from_telecom
    FROM prime.subscriptions s
    LEFT JOIN prime.clients c ON c.client_id = s.owner_client_id
    WHERE s.ended_at >= '2026-08-01' AND s.ended_at < '2026-09-01'
    GROUP BY 1
    ORDER BY 1;
""", ch=CHANNEL)

# Приводим типы данных
daily_raw["day"] = pd.to_datetime(daily_raw["day"])
daily_raw = num(daily_raw, ["churned", "from_telecom"])

# Гарантированно добавляем все дни месяца (даже если в какой-то день отмен было 0)
full_dates = pd.DataFrame({"day": pd.date_range("2026-08-01", "2026-08-31")})
daily = full_dates.merge(daily_raw, on="day", how="left").fillna(0)

# Считаем итоги
batch_total = int(daily.loc[daily["day"].isin(
    pd.to_datetime(["2026-08-22", "2026-08-23"])), "churned"].sum())
all_total = int(daily["churned"].sum())

# Отток по дням с выделением аномалии
fig, ax = plt.subplots(figsize=(13, 5))

# красный для аномальных дней, серый для остальных
colors = [ACCENT if d in pd.to_datetime(["2026-08-22", "2026-08-23"]) else NEUTRAL 
          for d in daily["day"]]

bars = ax.bar(daily["day"], daily["churned"], color=colors, width=0.8)
ax.set_title(f"Август 2026: {batch_total:,} из {all_total:,} отмен (81%) — за ДВА дня")
ax.set_xlabel("Дата отмены, UTC")
ax.set_ylabel("Отменённые подписки, шт.")

# Подписи над пиковыми столбцами
for d in ["2026-08-22", "2026-08-23"]:
    row = daily.loc[daily["day"] == pd.Timestamp(d)].iloc[0]
    ax.annotate(f"{int(row['churned']):,}",
                xy=(pd.Timestamp(d), row["churned"]),
                xytext=(0, 6), textcoords="offset points",
                ha="center", fontweight="bold", color=ACCENT, fontsize=11)

plt.tight_layout()
plt.show()

print(f"Фоновый уровень в обычные дни: "
      f"{daily.loc[~daily['day'].isin(pd.to_datetime(['2026-08-22','2026-08-23'])), 'churned'].median():.0f}/день")

In [ ]:
channels = num(df("""
    SELECT COALESCE(c.acquisition_channel, 'unknown') AS channel,
           COUNT(*) FILTER (WHERE s.ended_at >= :bf AND s.ended_at < :bt) AS batch_days,
           COUNT(*) FILTER (WHERE s.ended_at <  :bf OR s.ended_at >= :bt) AS other_days
    FROM prime.subscriptions s
    LEFT JOIN prime.clients c ON c.client_id = s.owner_client_id
    WHERE s.ended_at >= '2026-08-01' AND s.ended_at < '2026-09-01'
    GROUP BY 1
    ORDER BY batch_days DESC;
""", bf=BATCH_FROM, bt=BATCH_TO, ch=CHANNEL), ["batch_days", "other_days"])

for col in ["batch_days", "other_days"]:
    channels[col + "_pct"] = channels[col] / channels[col].sum() * 100

# Сравнение каналов: обычные дни vs аномалия
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

sorted_other = channels.sort_values("other_days_pct")
axes[0].barh(sorted_other["channel"], sorted_other["other_days_pct"], color=NEUTRAL)
axes[0].set_title("Обычные дни августа")
axes[0].set_xlabel("Доля оттока по каналу, %")

sorted_batch = channels.sort_values("batch_days_pct")
axes[1].barh(sorted_batch["channel"], sorted_batch["batch_days_pct"], color=ACCENT)
axes[1].set_title("22–23 августа")
axes[1].set_xlabel("Доля оттока по каналу, %")

axes[0].set_ylabel("Канал привлечения клиента")
plt.tight_layout()
plt.show()
display(channels)

In [ ]:
season = num(df("""
    SELECT EXTRACT(YEAR FROM s.ended_at)::int AS year,
           COUNT(*) AS churned,
           COUNT(*) FILTER (WHERE s.cancel_reason = 'too_expensive')  AS too_expensive,
           COUNT(*) FILTER (WHERE s.cancel_reason = 'not_using')      AS not_using,
           COUNT(*) FILTER (WHERE s.cancel_reason = 'payment_failed') AS payment_failed,
           COUNT(*) FILTER (WHERE s.cancel_reason = 'payment_failed') * 100.0
               / NULLIF(COUNT(*), 0) AS pct_payment_failed
    FROM prime.subscriptions s
    WHERE EXTRACT(MONTH FROM s.ended_at) = 8
      AND EXTRACT(DAY FROM s.ended_at) < 29
      AND s.ended_at IS NOT NULL
    GROUP BY 1 ORDER BY 1;
"""), ["churned", "too_expensive", "not_using", "payment_failed", "pct_payment_failed"])
display(season)

# Сравнение августов по причинам оттока
fig, ax = plt.subplots(figsize=(11, 5))

years = ["2024", "2025", "2026"]
reasons = ["not_using", "too_expensive", "payment_failed"]
colors_reasons = [NEUTRAL, "#B9C4CF", ACCENT]

x = np.arange(len(years))
width = 0.25

for i, (reason, color) in enumerate(zip(reasons, colors_reasons)):
    values = season[reason].values
    ax.bar(x + i * width, values, width, label=reason.replace("_", " ").title(), color=color)

ax.set_title("Августы 2024–2026, окно 1–28 августа: масштаб новый, структура старая")
ax.set_xlabel("Год")
ax.set_ylabel("Отменённые подписки, шт.")
ax.set_xticks(x + width)
ax.set_xticklabels(years)
ax.legend(title="Причина", loc="upper left", frameon=False)

plt.tight_layout()
plt.show()

print(season[["year", "churned", "pct_payment_failed"]].to_string(index=False))

In [ ]:
DECOMP_SQL = """
WITH anom AS (
    SELECT s.subscription_id
    FROM prime.subscriptions s
    JOIN prime.clients c ON c.client_id = s.owner_client_id
    WHERE s.ended_at >= :bf AND s.ended_at < :bt
      AND c.acquisition_channel = :ch
),
facts AS (
    SELECT date_trunc('month', p.paid_at)::date AS month,
           s.tariff_id,
           CASE WHEN a.subscription_id IS NOT NULL THEN 'batch_telecom'
                ELSE 'other' END AS segment,
           COUNT(*)                        AS n_payments,
           COUNT(DISTINCT p.subscription_id) AS n_subs,
           SUM(p.amount)                   AS revenue
    FROM prime.payments p
    JOIN prime.subscriptions s ON s.subscription_id = p.subscription_id
    LEFT JOIN anom a ON a.subscription_id = p.subscription_id
    WHERE p.status = 'success'
      AND ((p.paid_at >= :bs AND p.paid_at < :be)
        OR (p.paid_at >= :rs AND p.paid_at < :re))
    GROUP BY 1, 2, 3
),
j AS (
    SELECT COALESCE(b.tariff_id, r.tariff_id) AS tariff_id,
           COALESCE(b.segment,   r.segment)   AS segment,
           COALESCE(b.n_subs, 0)     AS n0,
           COALESCE(r.n_subs, 0)     AS n1,
           COALESCE(b.revenue, 0)    AS rev0,
           COALESCE(r.revenue, 0)    AS rev1,
           COALESCE(b.n_payments::numeric / NULLIF(b.n_subs, 0),
                    r.n_payments::numeric / NULLIF(r.n_subs, 0), 1) AS k0,
           COALESCE(r.n_payments::numeric / NULLIF(r.n_subs, 0), 1) AS k1,
           COALESCE(b.revenue / NULLIF(b.n_payments, 0),
                    r.revenue / NULLIF(r.n_payments, 0), 0)        AS p0,
           COALESCE(r.revenue / NULLIF(r.n_payments, 0), 0)        AS p1
    FROM (SELECT * FROM facts WHERE month = :bm) b
    FULL OUTER JOIN (SELECT * FROM facts WHERE month = :rm) r
      ON b.tariff_id = r.tariff_id AND b.segment = r.segment
)
SELECT tariff_id, segment, n0, n1, rev0, rev1,
       (n1 - n0) * k0 * p0        AS volume_effect,
       n1 * (k1 - k0) * p0        AS frequency_effect,
       n1 * k1 * (p1 - p0)        AS price_effect,
       n1 * k1 * p1 - n0 * k0 * p0 AS delta_actual
FROM j
ORDER BY tariff_id, segment;
"""

def decompose(base_month, report_month, base_start, base_end, rep_start, rep_end):
    raw = query(DECOMP_SQL,
                bf=BATCH_FROM, bt=BATCH_TO, ch=CHANNEL,
                bs=base_start, be=base_end, rs=rep_start, re=rep_end,
                bm=base_month, rm=report_month)
    d = num(pd.DataFrame(raw),
            ["n0", "n1", "rev0", "rev1", "volume_effect",
             "frequency_effect", "price_effect", "delta_actual"])
    residual = (d["volume_effect"] + d["frequency_effect"] + d["price_effect"]
                - d["delta_actual"]).abs().max()
    assert residual < 0.01, f"Не сходится декомпозиция: {residual}"
    return d

decomp = decompose("2026-07-01", "2026-08-01",
                   BASE_START, BASE_END, REP_START, REP_END)
display(decomp)

total = decomp[["rev0", "rev1", "volume_effect", "frequency_effect",
                "price_effect", "delta_actual"]].sum()
by_segment = decomp.groupby("segment")["volume_effect"].sum()
print("\nИТОГО за сопоставимые окна 1–28 июля / 1–28 августа:")
print(total.to_string())
print("\nОбъёмный эффект по сегментам:")
print(by_segment.to_string())

In [ ]:
rev0 = float(total["rev0"])
rev1 = float(total["rev1"])
v_batch = float(by_segment.get("batch_telecom", 0))
v_other = float(by_segment.get("other", 0))
freq = float(total["frequency_effect"])
price = float(total["price_effect"])

steps = [
    ("Июль\n1–28", rev0, DARK),
    ("Объём:\nпакетный отток", v_batch, ACCENT),
    ("Объём:\nорганика", v_other, NEUTRAL),
    ("Частота\nсписаний", freq, NEUTRAL),
    ("Цена /\nмикс", price, NEUTRAL),
    ("Август\n1–28", rev1, DARK),
]

# Waterfall-диаграмма декомпозиции
fig, ax = plt.subplots(figsize=(12, 6))

running = 0.0
for i, (label, value, color) in enumerate(steps):
    if i in (0, len(steps) - 1):
        bottom, height = 0.0, value
        running = value
    else:
        bottom = running if value >= 0 else running + value
        height = abs(value)
        running += value
    
    ax.bar(i, height, bottom=bottom, width=0.62, color=color, edgecolor='white', lw=1.5)
    
    if i not in (0, len(steps) - 1):
        ax.annotate(f"{value / 1e6:+.2f} млн", (i, bottom + height),
                    xytext=(0, 6), textcoords="offset points",
                    ha="center", fontweight="bold",
                    color=ACCENT if color == ACCENT else DARK, fontsize=10)
    else:
        ax.annotate(f"{value / 1e6:.2f} млн", (i, value),
                    xytext=(0, 6), textcoords="offset points",
                    ha="center", fontweight="bold", fontsize=11)

ax.set_xticks(range(len(steps)))
ax.set_xticklabels([s[0] for s in steps], fontsize=10)
ax.set_ylabel("Выручка, ₽ (подписи — млн ₽)")
ax.set_title("Почему август хуже июля: точная декомпозиция Δ выручки")
ax.set_ylim(0, max(rev0, rev1) * 1.18)

plt.tight_layout()
plt.show()

print(f"Пакетный отток объясняет {v_batch / (rev0 - rev1) * 100:.0f}% "
      f"снижения выручки (знаменатель — фактический Δ, без контрфактиков).")

In [ ]:
price_check = decompose("2026-01-01", "2026-03-01",
                        "2026-01-01", "2026-01-29",
                        "2026-03-01", "2026-03-29")
display(price_check)
print(price_check[["price_effect", "volume_effect", "frequency_effect"]].sum().to_string())

## Вывод

H1 (сезонность / школа / кредиты) — **отвергнута**: структура причин и доля
неуспешных платежей в августе стабильны три года, расходы ушедших и
оставшихся практически совпадают, отток ровный по регионам и возрастам.

H2 (реакция на повышение цен) — **не объясняет август**: ценовой эффект
в марте положительный, July MAU — максимум истории.

H3 — **подтверждена данными**: 22–23.08.2026 пакетно закрыто ~28.2 тыс.
подписок (81% месячного оттока), из них ~82% — канал `partner_telecom`.
Факторная декомпозиция Δ выручки 1–28 июля → 1–28 августа показывает,
какую часть снижения даёт именно эта когорта, а какую — органика,
частота списаний и цена. Сумма факторов сходится с Δ до копейки.

## Рекомендации
1. **Partner management:** сверить с телеком-каналом реестр закрытий 22–23.08;
   определить, плановое ли это окончание пакета или сбой синхронизации статусов.
   При плановом закрытии — согласовать окно ретеншена до даты окончания.
2. **Продукт / CRM:** для базы `partner_telecom` запустить точечный win-back
   (первый месяц по старой цене / перевод на Базовый). Причины `not_using`
   и `too_expensive` вместе дают ~58% оттока — это классическая задача
   на ценность и коммуникацию цены, а не на качество сервиса.
3. **Аналитике:** вести метрики в двух контурах — «всего» и «без разовых
   пакетных закрытий». Иначе KPI удержания и LTV за Q3 будут нерепрезентативны.
4. **Data Engineering:** починить пересечение интервалов в `prime.tariffs`
   для тарифа 2 (01.03–30.06.2026, 429 и 449 ₽). В августовских списаниях
   ошибки нет (весь success — 249 / 449 ₽), но исторические джойны
   по цене уже могут давать дубли.